In [ ]:
import torch
import torch.nn.functional as F
import time

## 2D Convolution (stride=1, pad=0)

`input[N][C][H][W] * weight[K][C][R][S] -> output[N][K][OH][OW]`

where `OH = H - R + 1`, `OW = W - S + 1`

In [ ]:
# 最基本的 conv2d 调用
inp = torch.randn(1, 16, 32, 32)
wt = torch.randn(32, 16, 3, 3)

out = F.conv2d(inp, wt, stride=1, padding=0)
print(f"input:  {list(inp.shape)}")
print(f"weight: {list(wt.shape)}")
print(f"output: {list(out.shape)}")  # [1, 32, 30, 30]

In [ ]:
# conv2d与手动实现的等价性验证
# 手动unfold实现：用 F.unfold 做 im2col
inp = torch.randn(2, 3, 8, 8)
wt = torch.randn(5, 3, 3, 3)

ref = F.conv2d(inp, wt, stride=1, padding=0)

# im2col: unfold 将 input 展开为 [N, C*R*S, OH*OW]
col = F.unfold(inp, kernel_size=3, stride=1, padding=0)
# weight: [K, C*R*S]
wt_flat = wt.view(wt.size(0), -1)
# matmul: [K, C*R*S] @ [C*R*S, N*OH*OW] -> [K, N*OH*OW]
out_matmul = wt_flat @ col
out_matmul = out_matmul.view(5, 2, 6, 6).transpose(0, 1)  # -> [N, K, OH, OW]

print(f"im2col+matmul max_err: {(out_matmul - ref).abs().max().item():.2e}")

In [ ]:
# 手动两种for循环实现conv，展示数据访问模式
def conv2d_naive(inp, wt):
    """朴素7重循环实现，用于理解数据流"""
    N, C, H, W = inp.shape
    K, _, R, S = wt.shape
    OH, OW = H - R + 1, W - S + 1
    out = torch.zeros(N, K, OH, OW)
    for n in range(N):
        for k in range(K):
            for oh in range(OH):
                for ow in range(OW):
                    for c in range(C):
                        for r in range(R):
                            for s in range(S):
                                out[n, k, oh, ow] += inp[n, c, oh+r, ow+s] * wt[k, c, r, s]
    return out

# 向量化版本：去掉内层循环 (C, R, S)
def conv2d_im2col(inp, wt):
    """im2col + matmul: 将卷积转为矩阵乘法"""
    N, C, H, W = inp.shape
    K, _, R, S = wt.shape
    OH, OW = H - R + 1, W - S + 1
    
    # im2col: 对每个输出位置展开一个 R×S×C 的列向量
    col = F.unfold(inp, kernel_size=(R, S), stride=1, padding=0)
    # weight flatten: [K, C*R*S]
    wt_flat = wt.view(K, -1)
    # GEMM
    out = wt_flat @ col  # [K, N*OH*OW]
    return out.view(K, N, OH, OW).transpose(0, 1)

inp = torch.randn(1, 3, 8, 8)
wt = torch.randn(4, 3, 3, 3)

o_naive = conv2d_naive(inp, wt)
o_im2col = conv2d_im2col(inp, wt)
o_torch = F.conv2d(inp, wt, stride=1, padding=0)

print(f"naive vs torch: {(o_naive - o_torch).abs().max():.2e}")
print(f"im2col vs torch: {(o_im2col - o_torch).abs().max():.2e}")

In [ ]:
# 其他conv参数: stride, padding, dilation, groups
inp = torch.randn(2, 16, 28, 28)
wt = torch.randn(32, 16, 3, 3)

# stride=2: 下采样
o1 = F.conv2d(inp, wt, stride=2, padding=0)
print(f"stride=2: {list(o1.shape)}")  # [2, 32, 13, 13]

# padding=1: 保持空间尺寸 (same padding for 3x3 kernel)
o2 = F.conv2d(inp, wt, stride=1, padding=1)
print(f"pad=1:   {list(o2.shape)}")   # [2, 32, 28, 28]

# dilation=2: 空洞卷积
o3 = F.conv2d(inp, wt, stride=1, padding=2, dilation=2)
print(f"dilate=2:{list(o3.shape)}")   # [2, 32, 28, 28]

# groups: 分组卷积 (depthwise conv when groups=C=K)
wt_dw = torch.randn(16, 1, 3, 3)  # [C, 1, R, S]
o4 = F.conv2d(inp, wt_dw, stride=1, padding=0, groups=16)
print(f"groups=16:{list(o4.shape)}")  # [2, 16, 26, 26]

In [ ]:
# 性能对比: torch.conv2d vs im2col+matmul vs naive
def bench_conv(inp, wt, fn, name, iters=100):
    # warmup
    for _ in range(10):
        fn(inp, wt)
    torch.cuda.synchronize()
    
    start = time.perf_counter()
    for _ in range(iters):
        fn(inp, wt)
    torch.cuda.synchronize()
    elapsed = (time.perf_counter() - start) / iters * 1000  # ms
    
    N, C, H, W = inp.shape
    K, _, R, S = wt.shape
    OH, OW = H - R + 1, W - S + 1
    gflops = 2.0 * N * K * OH * OW * C * R * S * 1e-9 / (elapsed / 1000)
    print(f"  {name:20s} {elapsed:8.4f} ms  {gflops:7.2f} GFLOPS")
    return gflops

inp = torch.randn(1, 64, 56, 56, device='cuda')
wt = torch.randn(64, 64, 3, 3, device='cuda')

print("Conv2D Performance (GPU):")
bench_conv(inp, wt, lambda x, w: F.conv2d(x, w, stride=1, padding=0), 'torch.conv2d')
bench_conv(inp, wt, lambda x, w: conv2d_im2col(x, w), 'im2col+matmul')
# naive is too slow on GPU, skip